# Reranking Strategies for Retrieval

In retrieval pipelines, the first-stage retriever (dense, sparse, or hybrid) optimizes for **recall** and pulls a wide candidate set. A reranker then re-orders that set with a more expensive but more accurate scoring function to maximize **precision at the top**.

This notebook implements and compares three reranking strategies on the same retrieved candidates:

1. **Cross-encoder reranking** (BERT-style pairwise scoring, self-hosted)
2. **Cohere Rerank API** (managed reranker)
3. **LLM-based reranking** with `gpt-4o-mini` (both pointwise and listwise variants)

Models used:
- Embeddings: `text-embedding-3-small`
- LLM reranker and final answer: `gpt-4o-mini`
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`

The chunks retrieved at every stage are printed so the effect of each reranker is visible.

## 1. Install Dependencies

In [1]:
!pip install -q openai pinecone-client pypdf langchain langchain-text-splitters sentence-transformers cohere numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.5/350.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 27.8 MB/s eta 0:00:00


## 2. Load Secrets from Colab

Required Colab secrets (left sidebar, key icon):
- `OPENAI_API_KEY`
- `PINECONE_API_KEY`
- `COHERE_API_KEY` (optional, only needed for the Cohere Rerank section)

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
try:
    os.environ["COHERE_API_KEY"] = userdata.get("COHERE_API_KEY")
    cohere_available = True
except Exception:
    cohere_available = False

print("OpenAI key loaded:  ", bool(os.environ.get("OPENAI_API_KEY")))
print("Pinecone key loaded:", bool(os.environ.get("PINECONE_API_KEY")))
print("Cohere key loaded:  ", cohere_available)

OpenAI key loaded:   True
Pinecone key loaded: True
Cohere key loaded:   True


## 3. Upload a PDF

In [3]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_path}")

Saving IIA HR Policy.pdf to IIA HR Policy.pdf
Uploaded: IIA HR Policy.pdf


## 4. Chunk the Document

We use the same chunking approach as Notebook 1 so comparisons are fair. Each chunk carries metadata: `source`, `page`, `chunk_index`, and `char_count`.

In [4]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

reader = PdfReader(pdf_path)
pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    if text.strip():
        pages.append({"page": page_number, "text": text})

print(f"Extracted {len(pages)} non-empty pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
ci = 0
for p in pages:
    for piece in splitter.split_text(p["text"]):
        chunks.append({
            "id": f"{pdf_path}-p{p['page']}-c{ci}",
            "text": piece,
            "metadata": {
                "source": pdf_path,
                "page": p["page"],
                "chunk_index": ci,
                "char_count": len(piece),
            },
        })
        ci += 1

print(f"Total chunks created: {len(chunks)}")
print("\nPreview of first 3 chunks:")
for c in chunks[:3]:
    print("-" * 70)
    print(f"ID: {c['id']}")
    print(f"Metadata: {c['metadata']}")
    print(f"Text: {c['text'][:200]}{'...' if len(c['text']) > 200 else ''}")

Extracted 47 non-empty pages
Total chunks created: 159

Preview of first 3 chunks:
----------------------------------------------------------------------
ID: IIA HR Policy.pdf-p1-c0
Metadata: {'source': 'IIA HR Policy.pdf', 'page': 1, 'chunk_index': 0, 'char_count': 196}
Text: IIA HR POLICY REVISION 1.1                                                                                 1  
 
 
 
 
 
 
 
 
 
 
 
 
       Indian Industries Association 
 
Human Resource Policy
----------------------------------------------------------------------
ID: IIA HR Policy.pdf-p2-c1
Metadata: {'source': 'IIA HR Policy.pdf', 'page': 2, 'chunk_index': 1, 'char_count': 449}
Text: IIA HR POLICY REVISION 1.1                                                                                 2  
Preface 
Indian Industries Association (IIA) started as U.P Chapter of NAYE in 1985 was 
...
----------------------------------------------------------------------
ID: IIA HR Policy.pdf-p2-c2
Metadata: {'source': 'IIA H

## 5. First-Stage Retrieval with Pinecone

We embed the chunks with `text-embedding-3-small`, store them in Pinecone, and pull a wide candidate pool (top-20). These 20 candidates are the input to every reranker below.

In [6]:
pip install -U pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 16.7 MB/s eta 0:00:00


In [7]:
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import time

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM = 1536

def embed_texts(texts, batch_size=64):
    out = []
    for i in range(0, len(texts), batch_size):
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch_size])
        out.extend([d.embedding for d in resp.data])
    return out

print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL}...")
embeddings = embed_texts([c["text"] for c in chunks])
print(f"Embeddings complete. Dimension: {len(embeddings[0])}")

Embedding 159 chunks with text-embedding-3-small...
Embeddings complete. Dimension: 1536


In [8]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
INDEX_NAME = "rerank-demo"

if INDEX_NAME not in [i["name"] for i in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
    print(f"Created index: {INDEX_NAME}")
else:
    print(f"Reusing existing index: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)

vectors = []
for c, e in zip(chunks, embeddings):
    meta = dict(c["metadata"])
    meta["text"] = c["text"]
    vectors.append({"id": c["id"], "values": e, "metadata": meta})

for i in range(0, len(vectors), 100):
    index.upsert(vectors=vectors[i:i + 100])

time.sleep(2)
print(f"Indexed: {index.describe_index_stats()['total_vector_count']} vectors")

Reusing existing index: rerank-demo
Indexed: 168 vectors


In [9]:
QUERY = "What does the document say about the main topic?"  # Replace with a query relevant to your PDF
CANDIDATE_K = 20  # how many candidates the first stage returns
FINAL_K = 5       # how many results each reranker keeps

def first_stage_retrieve(query, k=CANDIDATE_K):
    q = embed_texts([query])[0]
    res = index.query(vector=q, top_k=k, include_metadata=True)
    out = []
    for m in res["matches"]:
        out.append({
            "id": m["id"],
            "score": float(m["score"]),
            "text": m["metadata"]["text"],
            "page": m["metadata"].get("page"),
        })
    return out

candidates = first_stage_retrieve(QUERY, k=CANDIDATE_K)

print(f"First-stage candidates for query: {QUERY!r}")
print("=" * 80)
for i, c in enumerate(candidates, start=1):
    print(f"\n[Candidate {i}] cosine_sim={c['score']:.4f}  page={c['page']}  id={c['id']}")
    print(c["text"])

First-stage candidates for query: 'What does the document say about the main topic?'

[Candidate 1] cosine_sim=0.3320  page=3  id=Assignment_ Building a Simple RAG-Based Educational Chatbot.pdf-p3-c4
●  Document  loading  
●  Text  chunking  
●  Embedding  generation  
●  Vector  database  storage  
●  Similarity  retrieval  
●  Prompt  creation  
●  LLM  response  generation  
 
Task  2  —  Chunking  Strategy  
Select  the  most  suitable  chunking  strategy  for  educational  documents.  
You  must:  
●  Explain  your  chosen  chunking  strategy  
●  Justify  why  it  is  suitable  for  textbook-style  content  
●  Define  chunk  size  and  overlap  
●  Explain  how  chunking  impacts  retrieval  quality  
 
Task  3  —  Embedding  Model  and  Vector  Database  
Choose:

[Candidate 2] cosine_sim=0.3316  page=4  id=IIA HR Policy.pdf-p4-c9
achieve its’ goal, as a natural culmination of activities. 
It focuses on employees to have clear expectations on work requirements, 
compensation, w

## 6. Strategy 1: Cross-Encoder Reranking

A cross-encoder takes a `(query, document)` pair, encodes them **jointly** through a Transformer, and outputs a single relevance score. Unlike bi-encoders (which encode query and doc separately and compare with cosine), the query and document attend to each other token-by-token, which makes the score much more accurate at the cost of latency.

We use `cross-encoder/ms-marco-MiniLM-L-6-v2`, a small but well-trained model from the MS MARCO passage ranking dataset. Larger options like `ms-marco-MiniLM-L-12-v2` or `bge-reranker-base` improve quality further.

**How it works:**
- Score every (query, candidate) pair
- Sort by score descending
- Keep the top-K

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def cross_encoder_rerank(query, candidates, k=FINAL_K):
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)
    scored = [{**c, "ce_score": float(s)} for c, s in zip(candidates, scores)]
    return sorted(scored, key=lambda x: x["ce_score"], reverse=True)[:k]

ce_results = cross_encoder_rerank(QUERY, candidates, k=FINAL_K)

print(f"Cross-encoder reranked top-{FINAL_K} for: {QUERY!r}")
print("=" * 80)
for i, r in enumerate(ce_results, start=1):
    print(f"\n[Rank {i}] ce_score={r['ce_score']:.4f}  "
          f"first_stage_sim={r['score']:.4f}  page={r['page']}  id={r['id']}")
    print(r["text"])

## 7. Strategy 2: Cohere Rerank

Cohere's `rerank-english-v3.0` is a managed cross-encoder accessible through an API. It is a strong out-of-the-box baseline and removes the need to host a model yourself. The trade-off is per-call cost and an external dependency.

**How it works:**
- Send the query and the list of candidate texts to the API
- Cohere returns the top-N candidates with a `relevance_score` in `[0, 1]`

This cell is a no-op if no Cohere API key is set.

In [ ]:
if cohere_available:
    import cohere
    co = cohere.Client(os.environ["COHERE_API_KEY"])

    def cohere_rerank(query, candidates, k=FINAL_K):
        docs = [c["text"] for c in candidates]
        resp = co.rerank(
            model="rerank-english-v3.0",
            query=query,
            documents=docs,
            top_n=k,
        )
        out = []
        for r in resp.results:
            base = candidates[r.index]
            out.append({**base, "cohere_score": float(r.relevance_score)})
        return out

    co_results = cohere_rerank(QUERY, candidates, k=FINAL_K)

    print(f"Cohere reranked top-{FINAL_K} for: {QUERY!r}")
    print("=" * 80)
    for i, r in enumerate(co_results, start=1):
        print(f"\n[Rank {i}] cohere_score={r['cohere_score']:.4f}  "
              f"first_stage_sim={r['score']:.4f}  page={r['page']}  id={r['id']}")
        print(r["text"])
else:
    print("COHERE_API_KEY not set in Colab secrets, skipping this section.")
    co_results = None

## 8. Strategy 3: LLM-Based Reranking with gpt-4o-mini

LLM reranking sends the query and candidates to a general-purpose model and asks it to score or order them. This is the most **flexible** approach: the LLM can follow custom criteria (e.g. "prefer chunks that mention pricing"), apply reasoning, and explain its choices.

Two common patterns:

- **Pointwise**: score each `(query, doc)` separately. More calls, easier to parallelize, gives a per-document score.
- **Listwise**: give the LLM the full candidate list in one call and ask for an ordering. Cheaper (one call), but the model sees all candidates at once which can help it make relative judgments.

We implement both and compare them.

### 8a. Pointwise LLM Reranking

For each candidate, we ask the model to return a relevance score from 0 to 10. We use `response_format` JSON mode so the output is easy to parse.

In [ ]:
import json as _json

POINTWISE_SYSTEM = (
    "You are a relevance grader. Given a user query and a single passage, "
    "rate how well the passage answers the query on a scale from 0 (irrelevant) "
    "to 10 (directly and completely answers the query). "
    "Respond with strict JSON of the form {\"score\": <number>, \"reason\": \"<short reason>\"} "
    "and nothing else."
)

def llm_rerank_pointwise(query, candidates, k=FINAL_K, model="gpt-4o-mini"):
    scored = []
    for c in candidates:
        user = f"Query: {query}\n\nPassage: {c['text']}"
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": POINTWISE_SYSTEM},
                      {"role": "user", "content": user}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        parsed = _json.loads(resp.choices[0].message.content)
        scored.append({
            **c,
            "llm_score": float(parsed.get("score", 0)),
            "llm_reason": parsed.get("reason", ""),
        })
    return sorted(scored, key=lambda x: x["llm_score"], reverse=True)[:k]

print("Running pointwise LLM rerank (one call per candidate, may take a minute)...")
llm_pointwise_results = llm_rerank_pointwise(QUERY, candidates, k=FINAL_K)

print(f"\nPointwise LLM reranked top-{FINAL_K} for: {QUERY!r}")
print("=" * 80)
for i, r in enumerate(llm_pointwise_results, start=1):
    print(f"\n[Rank {i}] llm_score={r['llm_score']:.2f}  page={r['page']}  id={r['id']}")
    print(f"Reason: {r['llm_reason']}")
    print(f"Text:\n{r['text']}")

### 8b. Listwise LLM Reranking

We send all candidates in one prompt and ask the model to return the top-K indices in order. This is roughly `N` times cheaper than pointwise (one call instead of `N`), at the cost of having to fit all candidates in the context window.

In [ ]:
LISTWISE_SYSTEM = (
    "You are a relevance ranker. Given a user query and a list of "
    "candidate passages, return the top-K passage indices ordered "
    "from most to least relevant. Respond with strict JSON of the "
    "form {\"order\": [int, int, ...]} and nothing else."
)

def llm_rerank_listwise(query, candidates, k=FINAL_K, model="gpt-4o-mini"):
    listing = []
    for i, c in enumerate(candidates):
        listing.append(f"[{i}] (page {c['page']}) {c['text']}")
    docs_block = "\n\n".join(listing)

    user = f"Query: {query}\n\nCandidates:\n{docs_block}\n\nReturn the top {k} indices."

    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": LISTWISE_SYSTEM},
                  {"role": "user", "content": user}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    parsed = _json.loads(resp.choices[0].message.content)
    order = parsed.get("order", [])

    out = []
    for rank, idx in enumerate(order[:k]):
        if 0 <= idx < len(candidates):
            out.append({**candidates[idx], "llm_rank": rank + 1})
    return out

print("Running listwise LLM rerank (single call)...")
llm_listwise_results = llm_rerank_listwise(QUERY, candidates, k=FINAL_K)

print(f"\nListwise LLM reranked top-{FINAL_K} for: {QUERY!r}")
print("=" * 80)
for r in llm_listwise_results:
    print(f"\n[Rank {r['llm_rank']}]  page={r['page']}  id={r['id']}  "
          f"first_stage_sim={r['score']:.4f}")
    print(r["text"])

## 9. Side-by-Side Comparison of Top-3 IDs

To see exactly how the rerankers differ on the same query, we print the top-3 chunk IDs from each method.

In [ ]:
def ids(lst, n=3):
    return [r["id"] for r in lst[:n]]

print(f"Query: {QUERY!r}\n")
print(f"First-stage      top-3: {ids(candidates)}")
print(f"Cross-encoder    top-3: {ids(ce_results)}")
if co_results is not None:
    print(f"Cohere           top-3: {ids(co_results)}")
print(f"LLM (pointwise)  top-3: {ids(llm_pointwise_results)}")
print(f"LLM (listwise)   top-3: {ids(llm_listwise_results)}")

# Show which chunks are unique to one method vs shared.
ce_top = set(ids(ce_results))
lw_top = set(ids(llm_listwise_results))
print(f"\nIn cross-encoder but not LLM-listwise: {ce_top - lw_top}")
print(f"In LLM-listwise but not cross-encoder: {lw_top - ce_top}")
print(f"Agreed on:                           {ce_top & lw_top}")

## 10. Generate the Final Answer with gpt-4o-mini

We pass the cross-encoder reranked chunks as context to `gpt-4o-mini` and ask it to answer the query, citing page numbers. You can swap `ce_results` for any of the other rerankers above to compare answers.

In [ ]:
def answer_with_context(query, retrieved):
    blocks = [f"[page {r['page']}, id {r['id']}]\n{r['text']}" for r in retrieved]
    ctx = "\n\n---\n\n".join(blocks)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content":
             "You are a careful assistant. Answer the user's question strictly "
             "from the provided context. If the answer is not in the context, "
             "say you don't know. Cite the page numbers you used."},
            {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {query}"},
        ],
        temperature=0.1,
    )
    return resp.choices[0].message.content

print("Final answer using cross-encoder reranked context:\n")
print(answer_with_context(QUERY, ce_results))

## 11. Optional: Delete the Pinecone Index

In [ ]:
# pc.delete_index(INDEX_NAME)
# print("Index deleted.")

## Summary of Reranking Strategies

| Strategy                | Quality   | Latency | Cost   | Self-host? | Notes |
|-------------------------|-----------|---------|--------|------------|-------|
| Cross-encoder           | High      | Medium  | Low    | Yes        | Best price-performance. Default reranker in many production RAG stacks. |
| Cohere Rerank           | High      | Low     | Medium | No         | Managed, no GPU. Very strong English quality. |
| LLM rerank (pointwise)  | Highest   | High    | High   | No         | One API call per candidate. Use when criteria are complex or per-doc reasoning matters. |
| LLM rerank (listwise)   | Very high | Medium  | Medium | No         | One API call for the whole list. Cheaper than pointwise, slightly less precise per-document. |

**When to use each:**

- **Cross-encoder**: the default for production RAG. Self-hosted, predictable cost, strong quality. Reach for it first.
- **Cohere Rerank**: when you want top-tier reranking without managing model infra, and the per-call cost fits your budget.
- **LLM pointwise**: when you need explanations per chunk, or when ranking criteria are complex enough that a small cross-encoder cannot capture them (e.g. "prefer passages that mention 2024 pricing for enterprise plans").
- **LLM listwise**: a good cost-quality compromise within LLM reranking. Watch the context window: at very large candidate pools, switch to pointwise or chunk the list.